In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# Computation of Powers via Iterative Squaring

As the powers of numbers get big fast, we use the module `Z` of the library Zarith, which implements integers of arbitrary size.  Their type is `Z.t`.

In [ ]:
#require "zarith.top";;

The function `power` takes two arguments:
- `m` is an integer of type `Z.t`, the *base*,
- `n` is a natural number of type `int`, the *exponent*.

The function returns the power $m^n$.  It would not make much sense to use the type `Z.t` for the exponent: If we really had an exponent $n$ such that $n > 2^{62} - 1$, then $m^n$ would not fit into the memory of the computer unless $m$ is an element of the set $\{-1, 0, 1\}$.

**Implementation:** The *recursive implementation* of the function `power` given below is based on the following two equations:
1. $m^0 = 1$
2. $m^n = \left\{\begin{array}{ll}
         m^{n/2} \cdot m^{n/2}          & \mbox{if $n$ is even};    \\
         m^{n/2} \cdot m^{n/2} \cdot m  & \mbox{if $n$ is odd}.
         \end{array}
         \right.
         $

Here, $n/2$ denotes the integer division of $n$ by $2$, which is computed by the operator `/` in OCaml.
Observe that we square the result `p` of the recursive invocation of `power`.
Hence this method is also known as *iterative squaring*.

`Z.one` is the number $1$ of type `Z.t`.  Inside the *local open* `Z.(...)`, the operator `*` denotes the multiplication of numbers of type `Z.t`.

In [ ]:
let rec power (m : Z.t) (n : int) : Z.t =
  if n = 0 then
    Z.one
  else
    let p = power m (n / 2) in
    if n mod 2 = 0 then
      Z.(p * p)
    else
      Z.(p * p * m)

The format specifier `%2d` prints an integer right-aligned in a column of width 2, and `%6s` prints a string right-aligned in a column of width 6.  The function `Z.to_string` converts a number of type `Z.t` into a string.

In [ ]:
for k = 0 to 10 do
  Printf.printf "3 ** %2d = %6s\n" k (Z.to_string (power (Z.of_int 3) k))
done

We compute $3^{500\,000}$.  This number has more than $200\,000$ digits.  Therefore, we only display the number of its digits.  The result `p` is defined locally with `let ... in`, since the toplevel would display all digits of a variable defined with `let` at the top level.

The function `time`, which is defined in the file `setup.ml`, takes two arguments:
- `label` is a string,
- `f` is a function whose only argument is `()`.

It computes `f ()`, prints `label` together with the time this computation took, and returns the result of `f ()`.  The computation to be measured is wrapped into an anonymous function `fun () -> ...`, so that it is only executed when `time` calls `f`.

In [ ]:
let p = time "power 3 500_000" (fun () -> power (Z.of_int 3) 500_000) in
String.length (Z.to_string p)

We compare the result with the result of the function `Z.pow` from the library Zarith, which is available as the operator `**` inside `Z.(...)`.  The expression `assert c` raises the exception `Assert_failure` if the condition `c` is `false`.  Otherwise, its value is `()`.  The function `Z.equal` checks whether two numbers of type `Z.t` are equal.

In [ ]:
assert (Z.equal Z.(~$3 ** 500_000) (power (Z.of_int 3) 500_000))